In [6]:
import time
import pandas as pd
from fol_solver import solve_fol

def process_dataset(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    print(f"\n==========================================")
    print(f" Processing: {dataset_name} ({len(df)} samples) ")
    print(f"==========================================")

    is_three_valued = False
    if "flag" in df.columns:
        unique_flags = set(df["flag"].astype(str).str.strip().unique())
        if "Uncertain" in unique_flags:
            is_three_valued = True

    predictions = []
    start_time = time.time()

    for idx, row in df.iterrows():
        ctx = str(row["context"])
        stmt = str(row["statement"])
        pred = solve_fol(ctx, stmt, is_three_valued=is_three_valued)
        predictions.append(pred)

    elapsed = time.time() - start_time
    df["predicted_label"] = predictions

    if "flag" in df.columns:
        df["flag"] = df["flag"].astype(str).str.strip()
        correct = (df["predicted_label"] == df["flag"]).sum()
        accuracy = (correct / len(df)) * 100
        print(f"Accuracy         : {correct} / {len(df)} ({accuracy:.2f}%)")
        print(f"Total Time       : {elapsed:.3f} s (Avg: {(elapsed/len(df))*1000:.2f} ms/row)")
        print("\nPrediction Distribution:")
        print(df["predicted_label"].value_counts().to_string())

    return df

if __name__ == "__main__":
    # 1. Dataset 1: RuleTaker (Synthetic Horn FOL)
    df1 = pd.read_csv("data/processed/ruletaker_fol_ood_test.csv")
    df1_res = process_dataset(df1, "ruletaker_fol_test")

    # 2. Dataset 2: CounterBench (Causal FOL)
    df2 = pd.read_csv("data/processed/counterbench_fol_train.csv")
    df2_res = process_dataset(df2, "counterbench_fol_train")

    # 3. Dataset 3: FOLIO (Full 3-Valued FOL)
    df3 = pd.read_csv("data/processed/folio_fol_validation.csv")
    df3_res = process_dataset(df3, "folio_fol_validation")

    def prepare_submission_df(df: pd.DataFrame, dataset_prefix: str = "") -> pd.DataFrame:
        id_col = None
        for col in df.columns:
            if col.lower() in ["id", "example_id", "sample_id", "index"]:
                id_col = col
                break

        if id_col is not None:
            ids = df[id_col].values
        else:
            ids = list(range(len(df)))

        return pd.DataFrame({
            "ID": ids,
            "Prediction": df["predicted_label"].values
        })

    sub1 = prepare_submission_df(df1_res)
    sub2 = prepare_submission_df(df2_res)
    sub3 = prepare_submission_df(df3_res)

    combined_submission = pd.concat([sub1, sub2, sub3], ignore_index=True)

    combined_submission.to_csv("submission.csv", index=False)

    print("\n==========================================")
    print(" SUBMISSION GENERATED ")
    print("==========================================")
    print(f"Total Rows Saved : {len(combined_submission)}")
    print(f"File Saved       : submission.csv")


 Processing: ruletaker_fol_test (375 samples) 
Accuracy         : 279 / 375 (74.40%)
Total Time       : 0.171 s (Avg: 0.46 ms/row)

Prediction Distribution:
predicted_label
False    322
True      53

 Processing: counterbench_fol_train (1200 samples) 
Accuracy         : 680 / 1200 (56.67%)
Total Time       : 0.366 s (Avg: 0.31 ms/row)

Prediction Distribution:
predicted_label
False    1025
True      175

 Processing: folio_fol_validation (203 samples) 
Accuracy         : 122 / 203 (60.10%)
Total Time       : 44.391 s (Avg: 218.67 ms/row)

Prediction Distribution:
predicted_label
Uncertain    125
True          50
False         28

 SUBMISSION GENERATED 
Total Rows Saved : 1778
File Saved       : submission.csv
